# Imports

In [1]:
import os
import optuna
import pickle

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from category_encoders import CatBoostEncoder

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [4]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Machine Learning | Pure CatBoost

In [5]:
cv_results = cross_val_score(
    estimator=CatBoostClassifier(
        auto_class_weights='Balanced',
        eval_metric='AUC',
        loss_function='Logloss',
        thread_count=1,
        verbose=0
    ),
    X=X_train, 
    y=y_train.PitNextLap,
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1,
    params={'cat_features': ['driver', 'compound', 'race']}
)

In [6]:
cv_results.mean()

np.float64(0.9478799492521249)

In [7]:
model = CatBoostClassifier(
    auto_class_weights='Balanced',
    thread_count=1,
    verbose=0
).fit(X_train, y_train.PitNextLap, cat_features=['driver', 'compound', 'race'])

## Feature Selection

In [8]:
perm_result = permutation_importance(
    estimator=model, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=-1,
    scoring='roc_auc'
)

importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [9]:
importance_df.query("importance_mean <= 0").feature.tolist()

['treeraceprogress_laptime_s',
 'treeposition',
 'treelaptime_s',
 'treeraceprogress_position',
 'treeposition_laptime_s',
 'lapnumber_low']

In [10]:
features_to_drop = importance_df.query("importance_mean <= 0").feature.tolist()

In [12]:
model_tuned = make_pipeline(
    DropFeatures(features_to_drop),
    CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='AUC',
        auto_class_weights="Balanced",
        verbose=0,
        thread_count=1,
        cat_features=['driver', 'compound', 'race']
    )
).fit(X_train, y_train)

# Deploy

In [13]:
dump_pickle(model_tuned, '../models/model_pure_catboost.pkl')

# Machine Learning | CatBoostEncoder + CatBoost

In [16]:
cv_results = cross_val_score(
    estimator=make_pipeline(
        CatBoostEncoder(cols=['driver', 'compound', 'race']),
        CatBoostClassifier(
            auto_class_weights='Balanced',
            eval_metric='AUC',
            loss_function='Logloss',
            thread_count=1,
            verbose=0
        ),
    ),
    X=X_train, 
    y=y_train.PitNextLap,
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1,
)

In [17]:
cv_results.mean()

np.float64(0.9464863062095595)

In [19]:
pipe_cat = make_pipeline(
    CatBoostEncoder(cols=['driver', 'compound', 'race']),
    CatBoostClassifier(
        auto_class_weights='Balanced',
        eval_metric='AUC',
        loss_function='Logloss',
        thread_count=1,
        verbose=0
    ),
).fit(X_train, y_train.PitNextLap)

## Feature Selection

In [20]:
perm_result = permutation_importance(
    estimator=pipe_cat, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=-1,
    scoring='roc_auc'
)

importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [21]:
importance_df.query("importance_mean <= 0").feature.tolist()

['treelaptime_s', 'treeraceprogress_position', 'treeposition_laptime_s']

In [22]:
pipe_tuned = make_pipeline(
    DropFeatures(features_to_drop),
    CatBoostEncoder(cols=['driver', 'compound', 'race']),
    CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='AUC',
        auto_class_weights="Balanced",
        verbose=0,
        thread_count=1
    )
).fit(X_train, y_train)

## Deploy

In [23]:
dump_pickle(pipe_tuned, '../models/model_catboostencoder_catboost.pkl')